In [7]:
from pathlib import Path
import subprocess
import pandas as pd
import tempfile

# Paths 
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")
DATA_PATH = BASE_DIR / "TAME"

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

# =========================
# Run only Original or with Wiener filtered audio, switch with TRUE of FALSE
RUN_ORIGINAL = True
RUN_WIENER = True

# check 
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("RUN_ORIGINAL:", RUN_ORIGINAL)
print("RUN_WIENER:", RUN_WIENER)

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not RUN_ORIGINAL and not RUN_WIENER:
    raise ValueError("Both RUN_ORIGINAL and RUN_WIENER are False. Nothing to process.")


# =========================
# Dataset config
# =========================
OPENSMILE_SETS = {}

if RUN_ORIGINAL:
    OPENSMILE_SETS.update({
        "original_baseline": {
            "input_root": DATA_PATH / "selected_original_audio_20_pain_balanced",
            "folders": {
                "": "original"
            },
            "suffix": "pain_original"
        },
        "original_intensity": {
            "input_root": DATA_PATH / "original_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB",
                "-3dB": "intensity_-3dB",
                "+3dB": "intensity_3dB",
                "+6dB": "intensity_6dB",
            },
            "suffix": "pain_original"
        },
        "original_gaussian": {
            "input_root": DATA_PATH / "original_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian",
                "medium_gaussian_noise": "medium_gaussian",
                "high_gaussian_noise": "high_gaussian",
                "very_high_gaussian_noise": "very_high_gaussian",
            },
            "suffix": "pain_original"
        },
        "original_pink": {
            "input_root": DATA_PATH / "original_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink",
                "medium_pink_noise": "medium_pink",
                "high_pink_noise": "high_pink",
                "very_high_pink_noise": "very_high_pink",
            },
            "suffix": "pain_original"
        },
        "original_lowpass": {
            "input_root": DATA_PATH / "original_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass",
                "medium_lowpass": "medium_lowpass",
                "high_lowpass": "high_lowpass",
                "very_high_lowpass": "very_high_lowpass",
            },
            "suffix": "pain_original"
        },
    })

if RUN_WIENER:
    OPENSMILE_SETS.update({
        "wiener_baseline": {
            "input_root": DATA_PATH / "selected_wiener_audio_20_pain_balanced",
            "folders": {
                "": "wiener"
            },
            "suffix": "pain"
        },
        "wiener_intensity": {
            "input_root": DATA_PATH / "wiener_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB",
                "-3dB": "intensity_-3dB",
                "+3dB": "intensity_3dB",
                "+6dB": "intensity_6dB",
            },
            "suffix": "pain"
        },
        "wiener_gaussian": {
            "input_root": DATA_PATH / "wiener_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian",
                "medium_gaussian_noise": "medium_gaussian",
                "high_gaussian_noise": "high_gaussian",
                "very_high_gaussian_noise": "very_high_gaussian",
            },
            "suffix": "pain"
        },
        "wiener_pink": {
            "input_root": DATA_PATH / "wiener_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink",
                "medium_pink_noise": "medium_pink",
                "high_pink_noise": "high_pink",
                "very_high_pink_noise": "very_high_pink",
            },
            "suffix": "pain"
        },
        "wiener_lowpass": {
            "input_root": DATA_PATH / "wiener_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass",
                "medium_lowpass": "medium_lowpass",
                "high_lowpass": "high_lowpass",
                "very_high_lowpass": "very_high_lowpass",
            },
            "suffix": "pain"
        },
    })


# =========================
# Functions
# =========================
def extract_opensmile_features(audio_root, output_features_csv, output_failed_csv):
    print(f"\n--- Processing folder: {audio_root} ---")

    if not audio_root.exists():
        raise FileNotFoundError(f"Audio root not found:\n{audio_root}")

    wav_files = sorted(audio_root.rglob("*.wav"))
    print(f"Number of wav files found: {len(wav_files)}")

    if len(wav_files) == 0:
        raise ValueError(f"No .wav files found in:\n{audio_root}")

    all_rows = []
    failed_files = []

    for i, wav_file in enumerate(wav_files, start=1):
        print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

        participant_id = wav_file.parent.name
        filename = wav_file.name

        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            temp_output_csv = Path(tmp.name)

        try:
            temp_output_csv.unlink(missing_ok=True)

            cmd = [
                str(OPENSMILE_EXE),
                "-C", str(CONFIG_FILE),
                "-I", str(wav_file),
                "-csvoutput", str(temp_output_csv),
            ]

            result = subprocess.run(cmd, capture_output=True, text=True)

            if result.returncode != 0:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "openSMILE return code != 0",
                    "stderr": result.stderr
                })
                continue

            if not temp_output_csv.exists():
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "no output made",
                    "stderr": result.stderr
                })
                continue

            try:
                feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
            except Exception as e:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": f"error in csv: {e}",
                    "stderr": result.stderr
                })
                continue

            if feature_df.empty:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "output csv is empty",
                    "stderr": result.stderr
                })
                continue

            row = feature_df.iloc[0].to_dict()
            row["name"] = filename
            row["participant_id"] = participant_id
            row["filename"] = filename
            row["file_path"] = str(wav_file)

            all_rows.append(row)

        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error: {e}",
                "stderr": ""
            })

        finally:
            temp_output_csv.unlink(missing_ok=True)

    features_df = pd.DataFrame(all_rows)

    if not features_df.empty:
        preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
        remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
        features_df = features_df[
            [col for col in preferred_front_cols if col in features_df.columns] + remaining_cols
        ]

    features_df.to_csv(output_features_csv, index=False)

    print("\nDone.")
    print("Number of successful feature rows:", len(features_df))
    print("Number of failed files:", len(failed_files))
    print("Features saved in:")
    print(output_features_csv)

    if failed_files:
        failed_df = pd.DataFrame(failed_files)
        failed_df.to_csv(output_failed_csv, index=False)
        print("\nFailed files saved in:")
        print(output_failed_csv)

    print("\nShape features dataframe:", features_df.shape)

    return features_df, failed_files


def make_output_paths(label, suffix):
    output_features_csv = BASE_DIR / f"opensmile_{label}_features_{suffix}.csv"
    output_failed_csv = BASE_DIR / f"opensmile_{label}_failed_files_{suffix}.csv"
    return output_features_csv, output_failed_csv


# =========================
# Run all selected sets
# =========================
results = {}

for set_name, settings in OPENSMILE_SETS.items():
    input_root = settings["input_root"]
    folder_dict = settings["folders"]
    suffix = settings["suffix"]

    print(f"\n========== {set_name.upper()} ==========")

    if not input_root.exists():
        raise FileNotFoundError(f"Input root not found:\n{input_root}")

    results[set_name] = {}

    for folder_name, label in folder_dict.items():
        audio_root = input_root if folder_name == "" else input_root / folder_name
        output_features_csv, output_failed_csv = make_output_paths(label, suffix)

        features_df, failed_files = extract_opensmile_features(
            audio_root=audio_root,
            output_features_csv=output_features_csv,
            output_failed_csv=output_failed_csv
        )

        results[set_name][label] = {
            "features_df": features_df,
            "failed_files": failed_files
        }

print("\nFinished processing all selected OpenSMILE sets.")

SMILExtract exists: True
Config exists: True
RUN_ORIGINAL: True
RUN_WIENER: True

========== ORIGINAL_BASELINE ==========

--- Processing folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_original_audio_20_pain_balanced ---
Number of wav files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Pro

OpenSMILE code only for the divided groups (to not run the code above again) 

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import tempfile

# =========================
# Paths
# =========================
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")
DATA_PATH = BASE_DIR / "TAME"

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

# =========================
# Settings
# =========================
RUN_ORIGINAL = True
RUN_WIENER = True
SKIP_IF_FEATURES_EXIST = True

print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("RUN_ORIGINAL:", RUN_ORIGINAL)
print("RUN_WIENER:", RUN_WIENER)
print("SKIP_IF_FEATURES_EXIST:", SKIP_IF_FEATURES_EXIST)

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not RUN_ORIGINAL and not RUN_WIENER:
    raise ValueError("Both RUN_ORIGINAL and RUN_WIENER are False. Nothing to process.")

# =========================
# Only NEW pain-group datasets
# =========================
OPENSMILE_SETS = {}

if RUN_ORIGINAL:
    OPENSMILE_SETS.update({
        "original_pain_group_1_to_4_baseline": {
            "input_root": DATA_PATH / "selected_original_audio_20_pain_group_1_to_4",
            "folders": {"": "original_pain_group_1_to_4"},
            "suffix": "pain_original"
        },
        "original_pain_group_1_to_4_intensity": {
            "input_root": DATA_PATH / "original_pain_group_1_to_4_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_1_to_4",
                "-3dB": "intensity_-3dB_pain_group_1_to_4",
                "+3dB": "intensity_3dB_pain_group_1_to_4",
                "+6dB": "intensity_6dB_pain_group_1_to_4",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_1_to_4_gaussian": {
            "input_root": DATA_PATH / "original_pain_group_1_to_4_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_1_to_4",
                "medium_gaussian_noise": "medium_gaussian_pain_group_1_to_4",
                "high_gaussian_noise": "high_gaussian_pain_group_1_to_4",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_1_to_4",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_1_to_4_pink": {
            "input_root": DATA_PATH / "original_pain_group_1_to_4_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_1_to_4",
                "medium_pink_noise": "medium_pink_pain_group_1_to_4",
                "high_pink_noise": "high_pink_pain_group_1_to_4",
                "very_high_pink_noise": "very_high_pink_pain_group_1_to_4",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_1_to_4_lowpass": {
            "input_root": DATA_PATH / "original_pain_group_1_to_4_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_1_to_4",
                "medium_lowpass": "medium_lowpass_pain_group_1_to_4",
                "high_lowpass": "high_lowpass_pain_group_1_to_4",
                "very_high_lowpass": "very_high_lowpass_pain_group_1_to_4",
            },
            "suffix": "pain_original"
        },

        "original_pain_group_5_to_6_baseline": {
            "input_root": DATA_PATH / "selected_original_audio_20_pain_group_5_to_6",
            "folders": {"": "original_pain_group_5_to_6"},
            "suffix": "pain_original"
        },
        "original_pain_group_5_to_6_intensity": {
            "input_root": DATA_PATH / "original_pain_group_5_to_6_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_5_to_6",
                "-3dB": "intensity_-3dB_pain_group_5_to_6",
                "+3dB": "intensity_3dB_pain_group_5_to_6",
                "+6dB": "intensity_6dB_pain_group_5_to_6",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_5_to_6_gaussian": {
            "input_root": DATA_PATH / "original_pain_group_5_to_6_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_5_to_6",
                "medium_gaussian_noise": "medium_gaussian_pain_group_5_to_6",
                "high_gaussian_noise": "high_gaussian_pain_group_5_to_6",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_5_to_6",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_5_to_6_pink": {
            "input_root": DATA_PATH / "original_pain_group_5_to_6_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_5_to_6",
                "medium_pink_noise": "medium_pink_pain_group_5_to_6",
                "high_pink_noise": "high_pink_pain_group_5_to_6",
                "very_high_pink_noise": "very_high_pink_pain_group_5_to_6",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_5_to_6_lowpass": {
            "input_root": DATA_PATH / "original_pain_group_5_to_6_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_5_to_6",
                "medium_lowpass": "medium_lowpass_pain_group_5_to_6",
                "high_lowpass": "high_lowpass_pain_group_5_to_6",
                "very_high_lowpass": "very_high_lowpass_pain_group_5_to_6",
            },
            "suffix": "pain_original"
        },

        "original_pain_group_7_to_10_baseline": {
            "input_root": DATA_PATH / "selected_original_audio_20_pain_group_7_to_10",
            "folders": {"": "original_pain_group_7_to_10"},
            "suffix": "pain_original"
        },
        "original_pain_group_7_to_10_intensity": {
            "input_root": DATA_PATH / "original_pain_group_7_to_10_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_7_to_10",
                "-3dB": "intensity_-3dB_pain_group_7_to_10",
                "+3dB": "intensity_3dB_pain_group_7_to_10",
                "+6dB": "intensity_6dB_pain_group_7_to_10",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_7_to_10_gaussian": {
            "input_root": DATA_PATH / "original_pain_group_7_to_10_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_7_to_10",
                "medium_gaussian_noise": "medium_gaussian_pain_group_7_to_10",
                "high_gaussian_noise": "high_gaussian_pain_group_7_to_10",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_7_to_10",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_7_to_10_pink": {
            "input_root": DATA_PATH / "original_pain_group_7_to_10_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_7_to_10",
                "medium_pink_noise": "medium_pink_pain_group_7_to_10",
                "high_pink_noise": "high_pink_pain_group_7_to_10",
                "very_high_pink_noise": "very_high_pink_pain_group_7_to_10",
            },
            "suffix": "pain_original"
        },
        "original_pain_group_7_to_10_lowpass": {
            "input_root": DATA_PATH / "original_pain_group_7_to_10_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_7_to_10",
                "medium_lowpass": "medium_lowpass_pain_group_7_to_10",
                "high_lowpass": "high_lowpass_pain_group_7_to_10",
                "very_high_lowpass": "very_high_lowpass_pain_group_7_to_10",
            },
            "suffix": "pain_original"
        },
    })

if RUN_WIENER:
    OPENSMILE_SETS.update({
        "wiener_pain_group_1_to_4_baseline": {
            "input_root": DATA_PATH / "selected_wiener_audio_20_pain_group_1_to_4",
            "folders": {"": "wiener_pain_group_1_to_4"},
            "suffix": "pain"
        },
        "wiener_pain_group_1_to_4_intensity": {
            "input_root": DATA_PATH / "wiener_pain_group_1_to_4_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_1_to_4",
                "-3dB": "intensity_-3dB_pain_group_1_to_4",
                "+3dB": "intensity_3dB_pain_group_1_to_4",
                "+6dB": "intensity_6dB_pain_group_1_to_4",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_1_to_4_gaussian": {
            "input_root": DATA_PATH / "wiener_pain_group_1_to_4_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_1_to_4",
                "medium_gaussian_noise": "medium_gaussian_pain_group_1_to_4",
                "high_gaussian_noise": "high_gaussian_pain_group_1_to_4",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_1_to_4",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_1_to_4_pink": {
            "input_root": DATA_PATH / "wiener_pain_group_1_to_4_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_1_to_4",
                "medium_pink_noise": "medium_pink_pain_group_1_to_4",
                "high_pink_noise": "high_pink_pain_group_1_to_4",
                "very_high_pink_noise": "very_high_pink_pain_group_1_to_4",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_1_to_4_lowpass": {
            "input_root": DATA_PATH / "wiener_pain_group_1_to_4_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_1_to_4",
                "medium_lowpass": "medium_lowpass_pain_group_1_to_4",
                "high_lowpass": "high_lowpass_pain_group_1_to_4",
                "very_high_lowpass": "very_high_lowpass_pain_group_1_to_4",
            },
            "suffix": "pain"
        },

        "wiener_pain_group_5_to_6_baseline": {
            "input_root": DATA_PATH / "selected_wiener_audio_20_pain_group_5_to_6",
            "folders": {"": "wiener_pain_group_5_to_6"},
            "suffix": "pain"
        },
        "wiener_pain_group_5_to_6_intensity": {
            "input_root": DATA_PATH / "wiener_pain_group_5_to_6_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_5_to_6",
                "-3dB": "intensity_-3dB_pain_group_5_to_6",
                "+3dB": "intensity_3dB_pain_group_5_to_6",
                "+6dB": "intensity_6dB_pain_group_5_to_6",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_5_to_6_gaussian": {
            "input_root": DATA_PATH / "wiener_pain_group_5_to_6_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_5_to_6",
                "medium_gaussian_noise": "medium_gaussian_pain_group_5_to_6",
                "high_gaussian_noise": "high_gaussian_pain_group_5_to_6",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_5_to_6",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_5_to_6_pink": {
            "input_root": DATA_PATH / "wiener_pain_group_5_to_6_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_5_to_6",
                "medium_pink_noise": "medium_pink_pain_group_5_to_6",
                "high_pink_noise": "high_pink_pain_group_5_to_6",
                "very_high_pink_noise": "very_high_pink_pain_group_5_to_6",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_5_to_6_lowpass": {
            "input_root": DATA_PATH / "wiener_pain_group_5_to_6_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_5_to_6",
                "medium_lowpass": "medium_lowpass_pain_group_5_to_6",
                "high_lowpass": "high_lowpass_pain_group_5_to_6",
                "very_high_lowpass": "very_high_lowpass_pain_group_5_to_6",
            },
            "suffix": "pain"
        },

        "wiener_pain_group_7_to_10_baseline": {
            "input_root": DATA_PATH / "selected_wiener_audio_20_pain_group_7_to_10",
            "folders": {"": "wiener_pain_group_7_to_10"},
            "suffix": "pain"
        },
        "wiener_pain_group_7_to_10_intensity": {
            "input_root": DATA_PATH / "wiener_pain_group_7_to_10_intensity_perturbations_pain",
            "folders": {
                "-6dB": "intensity_-6dB_pain_group_7_to_10",
                "-3dB": "intensity_-3dB_pain_group_7_to_10",
                "+3dB": "intensity_3dB_pain_group_7_to_10",
                "+6dB": "intensity_6dB_pain_group_7_to_10",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_7_to_10_gaussian": {
            "input_root": DATA_PATH / "wiener_pain_group_7_to_10_gaussian_noise_perturbations_pain",
            "folders": {
                "low_gaussian_noise": "low_gaussian_pain_group_7_to_10",
                "medium_gaussian_noise": "medium_gaussian_pain_group_7_to_10",
                "high_gaussian_noise": "high_gaussian_pain_group_7_to_10",
                "very_high_gaussian_noise": "very_high_gaussian_pain_group_7_to_10",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_7_to_10_pink": {
            "input_root": DATA_PATH / "wiener_pain_group_7_to_10_pink_noise_perturbations_pain",
            "folders": {
                "low_pink_noise": "low_pink_pain_group_7_to_10",
                "medium_pink_noise": "medium_pink_pain_group_7_to_10",
                "high_pink_noise": "high_pink_pain_group_7_to_10",
                "very_high_pink_noise": "very_high_pink_pain_group_7_to_10",
            },
            "suffix": "pain"
        },
        "wiener_pain_group_7_to_10_lowpass": {
            "input_root": DATA_PATH / "wiener_pain_group_7_to_10_lowpass_perturbations_pain",
            "folders": {
                "low_lowpass": "low_lowpass_pain_group_7_to_10",
                "medium_lowpass": "medium_lowpass_pain_group_7_to_10",
                "high_lowpass": "high_lowpass_pain_group_7_to_10",
                "very_high_lowpass": "very_high_lowpass_pain_group_7_to_10",
            },
            "suffix": "pain"
        },
    })

# =========================
# Functions
# =========================
def extract_opensmile_features(audio_root, output_features_csv, output_failed_csv):
    print(f"\n--- Processing folder: {audio_root} ---")

    if not audio_root.exists():
        raise FileNotFoundError(f"Audio root not found:\n{audio_root}")

    wav_files = sorted(audio_root.rglob("*.wav"))
    print(f"Number of wav files found: {len(wav_files)}")

    if len(wav_files) == 0:
        raise ValueError(f"No .wav files found in:\n{audio_root}")

    all_rows = []
    failed_files = []

    for i, wav_file in enumerate(wav_files, start=1):
        print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

        participant_id = wav_file.parent.name
        filename = wav_file.name

        with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
            temp_output_csv = Path(tmp.name)

        try:
            temp_output_csv.unlink(missing_ok=True)

            cmd = [
                str(OPENSMILE_EXE),
                "-C", str(CONFIG_FILE),
                "-I", str(wav_file),
                "-csvoutput", str(temp_output_csv),
            ]

            result = subprocess.run(cmd, capture_output=True, text=True)

            if result.returncode != 0:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "openSMILE return code != 0",
                    "stderr": result.stderr
                })
                continue

            if not temp_output_csv.exists():
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "no output made",
                    "stderr": result.stderr
                })
                continue

            try:
                feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
            except Exception as e:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": f"error in csv: {e}",
                    "stderr": result.stderr
                })
                continue

            if feature_df.empty:
                failed_files.append({
                    "participant_id": participant_id,
                    "filename": filename,
                    "file_path": str(wav_file),
                    "reason": "output csv is empty",
                    "stderr": result.stderr
                })
                continue

            row = feature_df.iloc[0].to_dict()
            row["name"] = filename
            row["participant_id"] = participant_id
            row["filename"] = filename
            row["file_path"] = str(wav_file)

            all_rows.append(row)

        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error: {e}",
                "stderr": ""
            })

        finally:
            temp_output_csv.unlink(missing_ok=True)

    features_df = pd.DataFrame(all_rows)

    if not features_df.empty:
        preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
        remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
        features_df = features_df[
            [col for col in preferred_front_cols if col in features_df.columns] + remaining_cols
        ]

    features_df.to_csv(output_features_csv, index=False)

    print("\nDone.")
    print("Number of successful feature rows:", len(features_df))
    print("Number of failed files:", len(failed_files))
    print("Features saved in:")
    print(output_features_csv)

    if failed_files:
        failed_df = pd.DataFrame(failed_files)
        failed_df.to_csv(output_failed_csv, index=False)
        print("\nFailed files saved in:")
        print(output_failed_csv)

    print("\nShape features dataframe:", features_df.shape)

    return features_df, failed_files


def make_output_paths(label, suffix):
    output_features_csv = BASE_DIR / f"opensmile_{label}_features_{suffix}.csv"
    output_failed_csv = BASE_DIR / f"opensmile_{label}_failed_files_{suffix}.csv"
    return output_features_csv, output_failed_csv


# =========================
# Run all selected sets
# =========================
results = {}

for set_name, settings in OPENSMILE_SETS.items():
    input_root = settings["input_root"]
    folder_dict = settings["folders"]
    suffix = settings["suffix"]

    print(f"\n========== {set_name.upper()} ==========")

    if not input_root.exists():
        raise FileNotFoundError(f"Input root not found:\n{input_root}")

    results[set_name] = {}

    for folder_name, label in folder_dict.items():
        audio_root = input_root if folder_name == "" else input_root / folder_name
        output_features_csv, output_failed_csv = make_output_paths(label, suffix)

        if SKIP_IF_FEATURES_EXIST and output_features_csv.exists():
            print(f"Skipping existing file: {output_features_csv.name}")
            results[set_name][label] = {
                "features_df": pd.read_csv(output_features_csv),
                "failed_files": []
            }
            continue

        features_df, failed_files = extract_opensmile_features(
            audio_root=audio_root,
            output_features_csv=output_features_csv,
            output_failed_csv=output_failed_csv
        )

        results[set_name][label] = {
            "features_df": features_df,
            "failed_files": failed_files
        }

print("\nFinished processing new pain-group OpenSMILE sets.")

SMILExtract exists: True
Config exists: True
RUN_ORIGINAL: True
RUN_WIENER: True
SKIP_IF_FEATURES_EXIST: True

========== ORIGINAL_PAIN_GROUP_1_TO_4_BASELINE ==========

--- Processing folder: C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht\TAME\selected_original_audio_20_pain_group_1_to_4 ---
Number of wav files found: 200
[1/200] Processing: p10085.LC.1.161.wav
[2/200] Processing: p10085.LC.14.45.wav
[3/200] Processing: p10085.LC.23.156.wav
[4/200] Processing: p10085.LC.34.122.wav
[5/200] Processing: p10085.LC.41.99999.wav
[6/200] Processing: p10085.LC.5.134.wav
[7/200] Processing: p10085.LW.28.85.wav
[8/200] Processing: p10085.LW.36.99999.wav
[9/200] Processing: p10085.RC.17.160.wav
[10/200] Processing: p10085.RC.19.78.wav
[11/200] Processing: p10085.RC.3.58.wav
[12/200] Processing: p15965.LW.12.45.wav
[13/200] Processing: p15965.LW.16.99999.wav
[14/200] Processing: p15965.LW.18.118.wav
[15/200] Processing: p15965.LW.19.67.wav
[16/200] Process